In [ ]:
import math
import sklearn
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools import add_constant
import pyarrow.parquet as pq
import gc

### Датасет с отраслевой структурой

In [ ]:
df_sectors = pq.read_table(r"datasets/data_Y48423005_112_v20250918.parquet", columns=['region_name', 'mun_district', 'year', 'indicator_period', 'okved2', 'indicator_value'])


df_sectors=df_sectors.to_pandas()
df_sectors.head()

,region_name,mun_district,year,indicator_period,okved2,indicator_value
0,Алтайский край,Залесовский муниципальный округ,2022,Январь-сентябрь,Всего по обследуемым видам экономической деяте...,1313.0
1,Алтайский край,Залесовский муниципальный округ,2022,Январь-март,Раздел O Государственное управление и обеспече...,261.0
2,Алтайский край,Залесовский муниципальный округ,2022,Январь-декабрь,"Раздел D Обеспечение электрической энергией, г...",42.0
3,Алтайский край,Залесовский муниципальный округ,2022,Январь-декабрь,"Раздел G Торговля оптовая и розничная, ремонт ...",61.0
4,Алтайский край,Залесовский муниципальный округ,2022,Январь-сентябрь,Раздел Н Транспортировка и хранение,84.0


In [ ]:
df_sectors.okved2.value_counts()

,count
okved2,
Всего по обследуемым видам экономической деятельности,82940
Раздел P Образование,81951
"Раздел O Государственное управление и обеспечение военной безопасности, социальное обеспечение",80209
"Раздел G Торговля оптовая и розничная, ремонт автотранспортных средств и мотоциклов",73018
"Раздел R Деятельность в области культуры, спорта, организации досуга и развлечений",72571
Раздел Q Деятельность в области здравоохранения и социальных услуг,71306
"Раздел M Деятельность профессиональная, научная и техническая",62010
"Раздел D Обеспечение электрической энергией, газом и паром, кондиционирование воздуха",56333
Раздел Н Транспортировка и хранение,54731


In [ ]:
df_sectors.mun_district.nunique()

3023

### Потребительские безналичные расходы

In [ ]:
df_spend = pd.read_parquet(r"datasets/potrebitelskie_beznalicnye_rashody_na_urovne_munizipalnyh_obrazovanij.parquet")
df_spend.head()

,period,value,obs_status,source,category_15,mo,freq,decimals,unit_measure,unit_mult
0,2023-01-01,13815.0,A,Данные СберИндекса,Все категории,Бай-Тайгинский муниципальный район,Месяц,3,руб.,0
1,2023-02-01,15664.0,A,Данные СберИндекса,Все категории,Бай-Тайгинский муниципальный район,Месяц,3,руб.,0
2,2023-03-01,16037.0,A,Данные СберИндекса,Все категории,Бай-Тайгинский муниципальный район,Месяц,3,руб.,0
3,2023-04-01,15705.0,A,Данные СберИндекса,Все категории,Бай-Тайгинский муниципальный район,Месяц,3,руб.,0
4,2023-05-01,16080.0,A,Данные СберИндекса,Все категории,Бай-Тайгинский муниципальный район,Месяц,3,руб.,0


### Средние зарплаты

In [ ]:
cols = ['region_name', 'mun_level',
       'mun_district', 'municipality',
       'year', 'indicator_value', 'indicator_period', 'okved2']

In [ ]:
df_wage = pq.read_table(r"datasets/data_Y48423007_112_v20250918.parquet", columns=cols)
df_wage = df_wage.to_pandas()
df_wage.head()

,region_name,mun_level,mun_district,municipality,year,indicator_value,indicator_period,okved2
0,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,2022,39051.8,Январь-декабрь,Раздел O Государственное управление и обеспече...
1,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,2022,21066.7,Январь-март,Раздел C Обрабатывающие производства
2,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,2022,28937.5,Январь-декабрь,"Раздел Е Водоснабжение, водоотведение, организ..."
3,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,2022,91666.7,Январь-декабрь,Раздел L Деятельность по операциям с недвижимы...
4,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,2022,32625.0,Январь-июнь,"Раздел А Сельское, лесное хозяйство, охота, ры..."


### Население и городское население

In [ ]:
cols_popul = ['region_name', 'mun_district', 'mun_level', 'year', 'indicator_value', 'mest', 'indicator_period']
df_popul = pq.read_table(
    "datasets/data_Y48112027_112_v20250918.parquet",
    columns=cols_popul
).to_pandas()


In [ ]:
df_popul = df_popul[
    df_popul['mun_level'] == 'Муниципальное образование верхнего уровня'
].copy()

In [ ]:
df_popul.indicator_period.value_counts()

,count
indicator_period,
На 1 января,106029
Значение показателя за год,332


In [ ]:
df_popul.head()

,region_name,mun_district,mun_level,year,indicator_value,mest,indicator_period
0,Алтайский край,Залесовский муниципальный округ,Муниципальное образование верхнего уровня,2022,12979.0,Все население,На 1 января
1,Алтайский край,Залесовский муниципальный округ,Муниципальное образование верхнего уровня,2022,NaN,Городское население,На 1 января
2,Алтайский край,Залесовский муниципальный округ,Муниципальное образование верхнего уровня,2022,12979.0,Сельское население,На 1 января
3,Алтайский край,Залесовский муниципальный округ,Муниципальное образование верхнего уровня,2023,11444.0,Сельское население,На 1 января
4,Алтайский край,Залесовский муниципальный округ,Муниципальное образование верхнего уровня,2023,NaN,Городское население,На 1 января


In [ ]:
total = df_popul[df_popul['mest'] == 'Все население'][['region_name', 'mun_district', 'year', 'indicator_value']].copy()
total['indicator_value'] = total['indicator_value'].fillna(0)

urban = df_popul[df_popul['mest'] == 'Городское население'][['region_name', 'mun_district', 'year', 'indicator_value']]
urban = urban.rename(columns={'indicator_value': 'urban_pop'})
urban['urban_pop'] = urban['urban_pop'].fillna(0)

df_popul_new = pd.merge(total, urban, on=['region_name','mun_district', 'year'], how='inner')
df_popul_new['urban_share'] = df_popul_new['urban_pop'] / df_popul_new['indicator_value']

df_popul = df_popul_new[['region_name', 'mun_district', 'year', 'indicator_value', 'urban_share']]
df_popul.head()

,region_name,mun_district,year,indicator_value,urban_share
0,Алтайский край,Залесовский муниципальный округ,2022,12979.0,0.000000
1,Алтайский край,Залесовский муниципальный округ,2023,11444.0,0.000000
2,Алтайский край,Залесовский муниципальный округ,2024,11281.0,0.000000
3,Алтайский край,Залесовский муниципальный округ,2025,11043.0,0.000000
4,Алтайский край,Муниципальный округ Змеиногорский район,2025,15587.0,0.584461


In [ ]:
df_wage = df_wage[df_wage.okved2 == "Всего по обследуемым видам экономической деятельности"]
df_wage = df_wage[cols]
df_wage.head()

,region_name,mun_level,mun_district,municipality,year,indicator_value,indicator_period,okved2
7,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,2022,29336.1,Январь-март,Всего по обследуемым видам экономической деяте...
18,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,2022,31088.6,Январь-сентябрь,Всего по обследуемым видам экономической деяте...
22,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,2022,31264.4,Январь-июнь,Всего по обследуемым видам экономической деяте...
42,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,2022,32600.0,Январь-декабрь,Всего по обследуемым видам экономической деяте...
91,Алтайский край,Муниципальное образование верхнего уровня,Залесовский муниципальный округ,Залесовский муниципальный округ,2023,35315.2,Январь-март,Всего по обследуемым видам экономической деяте...


#### Названия некоторых муниципальных округов не совпадают в разных табличках, так что заменяем для дальнейшего джойна

In [ ]:
first = [
    "муниципальный округ Андреапольский",
    "городской округ Беловский",
    "городской округ Дмитровский",
    "муниципальный округ Хасынский",
    "городской округ Пышминский",
    "муниципальный округ Волотовский",
    "городской округ город Новомосковск",
    "городской округ Лыткарино",
    "городской округ город Новоульяновск",
    "городской округ город Ирбит",
    "городской округ Пущино",
    "городской округ город Великие Луки",
    "муниципальный округ Невельский",
    "муниципальный округ Косинский",
    "городской округ Полевской",
    "городской округ город Октябрьский",
    "городской округ город Кирсанов",
    "муниципальный округ Макушинский",
    "городской округ город Кизел",
    "муниципальный округ город Оленегорск",
    "городской округ Первоуральск",
    "городской округ Бисертский",
    "городской округ город Ульяновск",
    "городской округ город Новоалтайск",
    "городской округ Находкинский",
    "городской округ город Елец",
    "городской округ Углегорский",
    "городской округ Северодвинск",
    "городской округ Бронницы",
    "городской округ город Иркутск",
    "муниципальный округ Сарапульский район",
    "городской округ город Скопин",
    "городской округ город Димитровград",
    "городской округ Добрянский",
    "городской округ Туринский",
    "городской округ город Ефремов",
    "городской округ Ирбитское",
    "городской округ город Выкса",
    "городской округ город Алейск",
    "городской округ Рефтинский",
    "городской округ Карабашский",
    "городской округ Ангарский",
    "городской округ Сергиево-Посадский",
    "городской округ город Грозный",
    "городской округ город Коряжма",
    "городской округ город Белокуриха",
    "муниципальный округ Ягоднинский",
    "городской округ Ильинский",
    "городской округ Сорочинский",
    "муниципальный округ Сусуманский",
    "городской округ Тайгинский",
    "городской округ рабочий поселок Новогуровский",
    "муниципальный округ Ярский район",
    "городской округ Троицкий",
    "городской округ Карпинск",
    "городской округ город Торжок",
    "городской округ город Новодвинск",
    "городской округ Староуткинск",
    "городской округ Курильский",
    "городской округ Сосьвинский",
    "муниципальный округ Частинский",
    "городской округ Баксан",
    "городской округ город Стерлитамак",
    "муниципальный округ Кочевский",
    "муниципальный округ Калязинский",
    "городской округ Богородский",
    "муниципальный округ Тазовский район",
    "городской округ город Махачкала",
    "городской округ Южно-Курильский",
    "городской округ Южноуральский",
    "муниципальный округ Нестеровский",
    "городской округ город Волгореченск",
    "городской округ Партизанский",
    "городской округ город Шумерля",
    "городской округ город Каменск-Уральский",
    "городской округ Чебаркульский",
    "муниципальный округ Новоржевский",
    "городской округ Соликамский",
    "городской округ город Новосибирск",
    "городской округ Клин",
    "городской округ город Новотроицк",
    "муниципальный округ Каракулинский район",
    "городской округ Новолялинский",
    "городской округ город Бородино",
    "муниципальный округ Багратионовский",
    "городской округ город Ялуторовск",
    "городской округ город Салават",
    "городской округ город Омск",
    "муниципальный округ Березовский",
    "городской округ Анадырь",
    "муниципальный округ Зубцовский",
    "городской округ город Урюпинск",
    "городской округ город Можга",
    "муниципальный округ Вуктыл",
    "муниципальный округ Чарышский район",
    "городской округ Осинниковский",
    "муниципальный округ Кесовогорский",
    "муниципальный округ Увинский район",
    "городской округ город Шиханы",
    "городской округ город Боготол",
    "городской округ Электрогорск",
    "городской округ Киселевский",
    "городской округ Истра",
    "муниципальный округ Фировский",
    "муниципальный округ Молоковский",
    "городской округ город Биробиджан",
    "городской округ поселок Михайловский",
    "городской округ Новокузнецкий",
    "городской округ Прокопьевский",
    "муниципальный округ Селтинский район",
    "городской округ город Кострома",
    "городской округ Ясненский",
    "городской округ город Канск",
    "городской округ округ Муром",
    "муниципальный округ Пижанский",
    "муниципальный округ Уинский",
    "городской округ город Гусь-Хрустальный",
    "городской округ Тавдинский",
    "городской округ город Мегион",
    "городской округ Лысьвенский",
    "городской округ город Райчихинск",
    "городской округ Нытвенский",
    "муниципальный округ Кудымкарский",
    "муниципальный округ Кишертский",
    "муниципальный округ Можгинский район",
    "муниципальный округ Дебёсский район",
    "городской округ Домодедово",
    "городской округ Мамоновский",
    "муниципальный район имени Полины Осипенко",
    "городской округ Щёлково",
    "городской округ Можайский",
    "городской округ Луховицы",
    "муниципальный округ Лесной",
    "городской округ город Бор",
    "городской округ город Обнинск",
    "городской округ город Сарапул",
    "городской округ Охинский",
    "городской округ город Искитим",
    "городской округ город Комсомольск-на-Амуре",
    "городской округ Кыштымский",
    "муниципальный округ Бологовский",
    "городской округ город Салехард",
    "городской округ Серовский",
    "городской округ Чапаевск",
    "городской округ город Рязань",
    "муниципальный округ Торопецкий",
    "городской округ Воскресенск",
    "городской округ город Курган",
    "муниципальный округ Селижаровский",
    "муниципальный округ Граховский район",
    "городской округ Холмский",
    "муниципальный округ Ольский",
    "городской округ Тымовский",
    "городской округ город-курорт Железноводск",
    "городской округ город Братск",
    "городской округ Октябрьск",
    "городской округ Верхнесалдинский",
    "городской округ город Дивногорск",
    "городской округ город Аргун",
    "городской округ город Обь",
    "муниципальный округ Губахинский",
    "городской округ Карачаевский",
    "муниципальный округ Арбажский",
    "городской округ город Вятские Поляны",
    "городской округ город Свободный",
    "городской округ Лотошино",
    "городской округ Наро-Фоминский",
    "городской округ Чусовской",
    "городской округ Кашира",
    "городской округ Артинский",
    "городской округ Качканарский",
    "городской округ город Кизляр",
    "городской округ город Волжский",
    "муниципальный округ Краснознаменский",
    "городской округ город Рыбинск",
    "городской округ поселок Кедровый",
    "городской округ Кемеровский",
    "городской округ город Бузулук",
    "городской округ город Переславль-Залесский",
    "городской округ Верхнее Дуброво",
    "городской округ Соль-Илецкий",
    "муниципальный округ Струго-Красненский",
    "городской округ Мысковский",
    "городской округ город Арзамас",
    "муниципальный округ Питкярантский",
    "городской округ Воркута",
    "городской округ Люберцы",
    "городской округ город Сибай",
    "городской округ город Липецк",
    "муниципальный округ Сегежский",
    "муниципальный округ Ветлужский",
    "муниципальный округ Великоустюгский",
    "муниципальный округ Карагайский",
    "муниципальный округ Опочецкий",
    "городской округ Верхний Тагил",
    "муниципальный округ Гайнский",
    "муниципальный округ Кунгурский",
    "городской округ город Ноябрьск",
    "муниципальный округ Локнянский",
    "городской округ город Десногорск",
    "городской округ город Архангельск",
    "городской округ Верхнеуфалейский",
    "городской округ Миасский",
    "муниципальный округ Шарканский район",
    "городской округ город Заринск",
    "городской округ город Уварово",
    "городской округ город Орск",
    "муниципальный округ Воткинский район",
    "городской округ Краснотурьинск",
    "городской округ Вышневолоцкий",
    "городской округ Коломна",
    "муниципальный округ Ординский",
    "муниципальный округ Приуральский район",
    "городской округ Осинский",
    "городской округ город Губкинский",
    "муниципальный округ Пермский",
    "городской округ Пушкинский",
    "городской округ город Мурманск",
    "муниципальный округ Богородский",
    "городской округ город Избербаш",
    "городской округ город Тверь",
    "городской округ Дегтярск",
    "городской округ Тугулымский",
    "городской округ Камышловский",
    "городской округ город Кедровый",
    "городской округ город Орёл",
    "муниципальный округ Киясовский район",
    "городской округ Сыктывкар",
    "городской округ город Енисейск",
    "городской округ Владивостокский",
    "городской округ город Рубцовск",
    "городской округ Шаховская",
    "городской округ Саранск",
    "городской округ Нальчик",
    "городской округ город Кулебаки",
    "городской округ Поронайский",
    "муниципальный округ Рамешковский",
    "городской округ город Магас",
    "городской округ Ивдельский",
    "муниципальный округ Конаковский",
    "муниципальный округ Шимановский",
    "городской округ Александровск-Сахалинский район",
    "городской округ город Ханты-Мансийск",
    "муниципальный округ Мантуровский",
    "городской округ Пермский",
    "городской округ город Уфа",
    "городской округ Долгопрудный",
    "городской округ Осташковский",
    "городской округ Спасск-Дальний",
    "городской округ город Котельнич",
    "муниципальный округ Надымский район",
    "муниципальный округ Среднеканский",
    "городской округ город Псков",
    "муниципальный округ Унинский",
    "городской округ город Кизилюрт",
    "городской округ Сокольский",
    "городской округ город Ишим",
    "городской округ город Нижний Тагил",
    "городской округ Красноуральск",
    "городской округ город Усть-Илимск",
    "городской округ Кохма",
    "городской округ город Южно-Сухокумск",
    "городской округ Вичуга",
    "городской округ город Тула",
    "муниципальный округ Куединский",
    "городской округ город Медногорск",
    "городской округ Кировградский",
    "городской округ город Шадринск",
    "городской округ Пелым",
    "городской округ город Красноярск",
    "городской округ Тольятти",
    "городской округ город Чита",
    "городской округ Магнитогорский",
    "муниципальный округ Юкаменский район",
    "городской округ Навашинский",
    "городской округ Павловский Посад",
    "городской округ Корсаковский",
    "городской округ Заречный",
    "городской округ Режевской",
    "городской округ город Буй",
    "городской округ Суксунский",
    "городской округ Ленинский",
    "городской округ город Лесосибирск",
    "городской округ Сосновоборский",
    "муниципальный округ Балезинский район",
    "муниципальный округ Кизнерский район",
    "городской округ город Рассказово",
    "городской округ город Березники",
    "городской округ Верещагинский",
    "городской округ город Новочебоксарск",
    "муниципальный округ Лихославльский",
    "городской округ Ленинск-Кузнецкий",
    "городской округ Ступино",
    "городской округ Эгвекинот",
    "городской округ город-курорт Ессентуки",
    "муниципальный округ Якшур-Бодьинский район",
    "муниципальный округ Алнашский район",
    "муниципальный округ Хвойнинский",
    "муниципальный округ Ямальский район",
    "городской округ город Магадан",
    "городской округ Северо-Курильский",
    "городской округ город Ярославль",
    "городской округ город Назарово",
    "городской округ Удомельский",
    "городской округ Лобня",
    "городской округ город Минусинск",
    "городской округ город Яровое",
    "городской округ город Бийск",
    "муниципальный округ Малопургинский район",
    "городской округ город Хабаровск",
    "городской округ город Нефтекамск",
    "муниципальный округ город Славгород",
    "городской округ Юргинский",
    "городской округ Красногорск",
    "городской округ город Саянск",
    "муниципальный округ Кикнурский",
    "муниципальный округ Большесосновский",
    "городской округ город Ижевск",
    "городской округ Краснокамский",
    "городской округ Полысаевский",
    "городской округ Горноуральский",
    "городской округ Челябинский",
    "муниципальный округ Мурашинский",
    "муниципальный округ Ковдорский",
    "городской округ Одинцовский",
    "городской округ Гаринский",
    "муниципальный округ Озерский",
    "муниципальный округ Виноградовский",
    "муниципальный округ Красноселькупский район",
    "городской округ Голышмановский",
    "городской округ Алапаевское",
    "муниципальный округ Правдинский",
    "муниципальный район Город Краснокаменск и Краснокаменский район",
    "городской округ Ревда",
    "городской округ город Нижний Новгород",
    "городской округ Кинель",
    "городской округ Певек",
    "городской округ город Оренбург",
    "городской округ город Первомайск",
    "муниципальный округ Зеленоградский",
    "городской округ город Шарыпово",
    "городской округ Петрозаводский",
    "городской округ Светловский",
    "городской округ Невьянский",
    "муниципальный район Заполярный район",
    "городской округ Реутов",
    "муниципальный округ Суетский район",
    "городской округ Серебряные Пруды",
    "городской округ Кашинский",
    "городской округ город Невинномысск",
    "городской округ город Шарья",
    "городской округ Анжеро-Судженский",
    "городской округ город Сасово",
    "городской округ Жигулевск",
    "муниципальный округ Тисульский",
    "городской округ город Когалым",
    "городской округ город Стрежевой",
    "муниципальный округ Западнодвинский",
    "городской округ Орехово-Зуевский",
    "городской округ город Урай",
    "городской округ Солнечногорск",
    "городской округ Усть-Катавский",
]

second = [
    "Андреапольский муниципальный район (с 2020 г. преобразован в Андреапольский муниципальный округ)",
    "Беловский",
    "Дмитровский",
    "Хасынский городской округ (до 2023 г.)",
    "Пышминский",
    "Волотовский муниципальный район",
    "Город Новомосковск",
    "Лыткарино",
    "Город Новоульяновск",
    "Город Ирбит",
    "Пущино",
    "Город Великие Луки",
    "Невельский муниципальный район",
    "Косинский муниципальный округ",
    "Полевской",
    "Город Октябрьский",
    "Город Кирсанов",
    "Макушинский муниципальный район",
    "Город Кизел",
    "Город Оленегорск",
    "Первоуральск",
    "Бисертский",
    "Город Ульяновск",
    "Город Новоалтайск",
    "Находкинский",
    "Город Елец",
    "Углегорский муниципальный район",
    "Северодвинск / «Муниципальный округ «\"Город Северодвинск\"\"»",
    "Бронницы",
    "Город Иркутск",
    "Сарапульский муниципальный район",
    "Город Скопин",
    "Город Димитровград",
    "Добрянский",
    "Туринский",
    "Город Ефремов",
    "Ирбитское",
    "Город Выкса",
    "Город Алейск",
    "Рефтинский",
    "Карабашский",
    "Ангарский",
    "Сергиево-Посадский",
    "Город Грозный",
    "Город Коряжма",
    "Город Белокуриха",
    "Ягоднинский городской округ (до 2023 г.) / Ягоднинский",
    "Ильинский",
    "Сорочинский",
    "Сусуманский городской округ (до 2023 г.) / Сусуманский",
    "Тайгинский",
    "Рабочий поселок Новогуровский",
    "Ярский муниципальный район",
    "Троицкий",
    "Карпинск",
    "Город Торжок",
    "Город Новодвинск",
    "Староуткинск",
    "Курильский",
    "Сосьвинский",
    "Частинский муниципальный район / Частинский муниципальный округ",
    "Баксан",
    "Город Стерлитамак",
    "Кочевский муниципальный район",
    "Калязинский муниципальный район (с 2024 г. преобразован в Калязинский муниципальный округ)",
    "Богородский / Богородский муниципальный район",
    "Тазовский муниципальный район / Муниципальный округ Тазовский район",
    "Город Махачкала",
    "Южно-Курильский",
    "Южноуральский",
    "Нестеровский г. о. / Нестеровский",
    "Город Волгореченск",
    "Партизанский",
    "Город Шумерля",
    "Каменск-Уральский",
    "Чебаркульский",
    "Новоржевский / Новоржевский муниципальный район",
    "Соликамский / Соликамский муниципальный округ",
    "Город Новосибирск",
    "Клин",
    "Город Новотроицк",
    "Муниципальный округ Каракулинский район / Каракулинский муниципальный район",
    "Новолялинский",
    "Город Бородино",
    "Багратионовский / Багратионовский г. о",
    "Город Ялуторовск",
    "Город Салават",
    "Город Омск",
    "Березовский / Березовский муниципальный округ",
    "Анадырь",
    "Зубцовский муниципальный район (с 2023 г. преобразован в Зубцовский муниципальный округ) / Зубцовский",
    "Город Урюпинск",
    "Город Можга",
    "Вуктыл (до 1 января 2024 года) / Муниципальный округ Вуктыл",
    "Чарышский муниципальный район / Муниципальный округ Чарышский район",
    "Осинниковский",
    "Кесовогорский / Кесовогорский муниципальный район (с 2023 г. преобразован в Кесовогорский муниципальный округ)",
    "Увинский муниципальный район / Муниципальный округ Увинский район",
    "Город Шиханы",
    "Город Боготол",
    "Электрогорск",
    "Киселевский",
    "Истра",
    "Фировский муниципальный район (с 2024 г. преобразован в Фировский муниципальный округ) / Фировский",
    "Молоковский / Молоковский муниципальный район (с 2022 г. преобразован в Молоковский муниципальный округ)",
    "Город Биробиджан",
    "Поселок Михайловский",
    "Новокузнецкий",
    "Прокопьевский",
    "Селтинский муниципальный район / Муниципальный округ Селтинский район",
    "Город Кострома",
    "Ясненский",
    "Город Канск",
    "Округ Муром",
    "Пижанский / Пижанский муниципальный район",
    "Уинский муниципальный район / Уинский муниципальный округ",
    "Город Гусь-Хрустальный",
    "Тавдинский",
    "Мегион",
    "Лысьвенский / Лысьвенский муниципальный округ",
    "Город Райчихинск",
    "Нытвенский / Нытвенский муниципальный округ",
    "Кудымкарский / Кудымкарский муниципальный округ",
    "Кишертский муниципальный округ / Кишертский муниципальный район",
    "Можгинский муниципальный район / Муниципальный округ Можгинский район",
    "Дебесский муниципальный район / Муниципальный округ Дебёсский район",
    "Домодедово",
    "Мамоновский",
    "Муниципальный район имени Полины Осипенко",
    "Щёлково",
    "Можайский / Можайский (до 2019 г.)",
    "Луховицы / Луховицкий",
    "Лесной",
    "Город Бор",
    "Город Обнинск",
    "Город Сарапул",
    "Охинский",
    "Город Искитим",
    "Город Комсомольск-на-Амуре",
    "Кыштымский",
    "Бологовский муниципальный район (с 2024 г. преобразован в Бологовский муниципальный округ)",
    "Город Салехард",
    "Серовский",
    "Чапаевск",
    "Город Рязань",
    "Торопецкий муниципальный район (с 2024 г. преобразован в Торопецкий муниципальный округ)",
    "Воскресенск",
    "Город Курган",
    "Селижаровский / Селижаровский муниципальный район (с 2021 г. преобразован в Селижаровский муниципальный округ)",
    "Граховский муниципальный район / Муниципальный округ Граховский район",
    "Холмский",
    "Ольский / Ольский городской округ (до 2023 г.)",
    "Тымовский",
    "Город-курорт Железноводск",
    "Город Братск",
    "Октябрьск",
    "Верхнесалдинский",
    "Город Дивногорск",
    "Город Аргун",
    "Город Обь",
    "Губахинский / Городской округ Губахинский",
    "Карачаевский",
    "Арбажский / Арбажский муниципальный район",
    "Город Вятские Поляны",
    "Город Свободный",
    "Лотошино / Лотошинский",
    "Наро-Фоминский / Наро-Фоминский (до 2018 г.)",
    "Чусовской",
    "Кашира",
    "Артинский",
    "Качканарский",
    "Город Кизляр",
    "Город Волжский",
    "Краснознаменский / Краснознаменский г. о",
    "Город Рыбинск",
    "Город Кедровый / Поселок Кедровый",
    "Кемеровский",
    "Город Бузулук",
    "Город Переславль-Залесский",
    "Верхнее Дуброво",
    "Соль-Илецкий",
    "Струго-Красненский / Струго-Красненский муниципальный район",
    "Мысковский",
    "Город Арзамас",
    "Питкярантский муниципальный округ / Питкярантский муниципальный район",
    "Воркута / Муниципальный округ Воркута",
    "Люберцы / Люберецкий",
    "Город Сибай",
    "Город Липецк",
    "Сегежский муниципальный округ / Сегежский муниципальный район",
    "Ветлужский муниципальный округ / Ветлужский муниципальный район",
    "Великоустюгский муниципальный округ / Великоустюгский муниципальный район",
    "Карагайский муниципальный округ / Карагайский муниципальный район",
    "Опочецкий / Опочецкий муниципальный район",
    "Верхний Тагил",
    "Гайнский / Гайнский муниципальный округ",
    "Кунгурский / Кунгурский муниципальный округ / Городской округ город Кунгур",
    "Город Ноябрьск",
    "Локнянский / Локнянский муниципальный район",
    "Город Десногорск",
    "Город Архангельск",
    "Верхнеуфалейский",
    "Миасский",
    "Шарканский муниципальный район / Муниципальный округ Шарканский район",
    "Город Заринск",
    "Город Уварово",
    "Город Орск",
    "Воткинский муниципальный район / Муниципальный округ Воткинский район",
    "Краснотурьинск",
    "Вышневолоцкий / Вышневолоцкий муниципальный район (с 2020 г. объединен с городским округом г. Вышний Волочек)",
    "Коломна / Коломенский (до 2018 г.)",
    "Ординский / Ординский муниципальный округ / Ординский муниципальный район",
    "Приуральский муниципальный район / Муниципальный округ Приуральский район",
    "Осинский / Осинский муниципальный округ",
    "Город Губкинский",
    "Пермский / Пермский муниципальный округ / Пермский муниципальный район",
    "Пушкинский / Пушкинский (до 2020 г.)",
    "Город-герой Мурманск",
    "Богородский / Богородский муниципальный округ",
    "Город Избербаш",
    "Город Тверь",
    "Дегтярск",
    "Тугулымский",
    "Камышловский",
    "Город Кедровый",
    "Город Орёл",
    "Киясовский муниципальный район / Муниципальный округ Киясовский район",
    "Сыктывкар",
    "Город Енисейск",
    "Владивостокский",
    "Город Рубцовск",
    "Шаховская",
    "Саранск",
    "Нальчик",
    "Город Кулебаки",
    "Поронайский",
    "Рамешковский / Рамешковский муниципальный район (с 2022 г. преобразован в Рамешковский муниципальный округ)",
    "Город Магас",
    "Ивдельский",
    "Конаковский муниципальный район (с 2024 г. преобразован в Конаковский муниципальный округ)",
    "Шимановский муниципальный округ / Шимановский муниципальный район",
    "Александровск-Сахалинский район",
    "Ханты-Мансийск",
    "Мантуровский муниципальный округ / Мантуровский муниципальный район",
    "Пермский",
    "Город Уфа",
    "Долгопрудный",
    "Осташковский / Осташковский муниципальный район (с 2018 г. преобразован в Осташковский городской округ)",
    "Спасск-Дальний",
    "Город Котельнич",
    "Надымский муниципальный район / Муниципальный округ Надымский район",
    "Среднеканский / Среднеканский городской округ (до 2023 г.)",
    "Город Псков",
    "Унинский / Унинский муниципальный район",
    "Город Кизилюрт",
    "Сокольский / Сокольский муниципальный округ",
    "Город Ишим",
    "Город Нижний Тагил",
    "Красноуральск",
    "Город Усть-Илимск",
    "Кохма",
    "Город Южно-Сухокумск",
    "Вичуга",
    "Город Тула",
    "Куединский / Куединский муниципальный округ / Куединский муниципальный район",
    "Город Медногорск",
    "Кировградский",
    "Город Шадринск / Шадринский муниципальный район",
    "Пелым",
    "Город Красноярск",
    "Тольятти",
    "Город Чита",
    "Магнитогорский",
    "Юкаменский муниципальный район / Муниципальный округ Юкаменский район",
    "Навашинский",
    "Павлово-Посадский",
    "Корсаковский",
    "Заречный / ЗАТО город Заречный",
    "Режевской",
    "Город Буй",
    "Суксунский / Суксунский муниципальный округ / Суксунский муниципальный район",
    "Ленинский / Ленинский (до 2021 г.)",
    "Город Лесосибирск",
    "Сосновоборский",
    "Балезинский муниципальный район / Муниципальный округ Балезинский район",
    "Кизнерский муниципальный район / Муниципальный округ Кизнерский район",
    "Город Рассказово",
    "Город Березники / Муниципальный округ город Березники",
    "Верещагинский / Верещагинский муниципальный округ",
    "Город Новочебоксарск",
    "Лихославльский / Лихославльский муниципальный район (с 2022 г. преобразован в Лихославльский муниципальный округ)",
    "Ленинск-Кузнецкий",
    "Ступино / Ступинский",
    "Эгвекинот / Муниципальный округ Эгвекинот",
    "Город-курорт Ессентуки",
    "Якшур-Бодьинский муниципальный район / Муниципальный округ Якшур-Бодьинский район",
    "Алнашский муниципальный район / Муниципальный округ Алнашский район",
    "Хвойнинский / Хвойнинский муниципальный район",
    "Ямальский муниципальный район / Муниципальный округ Ямальский район",
    "Город Магадан",
    "Северо-Курильский",
    "Город Ярославль",
    "Город Назарово",
    "Удомельский",
    "Лобня",
    "Город Минусинск",
    "Город Яровое",
    "Город Бийск",
    "Малопургинский муниципальный район / Муниципальный округ Малопургинский район",
    "Город Хабаровск",
    "Город Нефтекамск",
    "Город Славгород / Муниципальный округ город Славгород",
    "Юргинский / Юргинский муниципальный округ",
    "Красногорск",
    "Город Саянск",
    "Кикнурский / Кикнурский муниципальный район",
    "Большесосновский / Большесосновский муниципальный район",
    "Город Ижевск",
    "Краснокамский / Краснокамский муниципальный округ",
    "Полысаевский",
    "Горноуральский",
    "Челябинский",
    "Мурашинский / Мурашинский муниципальный район",
    "Ковдорский муниципальный округ / Ковдорский район",
    "Одинцовский / Одинцовский (до 2020 г.)",
    "Гаринский",
    "Озерский / Озерский г. о.",
    "Виноградовский муниципальный округ / Виноградовский муниципальный район",
    "Красноселькупский муниципальный район / Муниципальный округ Красноселькупский район",
    "Голышмановский / Голышмановский муниципальный район",
    "Алапаевское",
    "Правдинский / Правдинский г. о.",
    "Муниципальный район Город Краснокаменск и Краснокаменский район",
    "Ревда",
    "Город Нижний Новгород",
    "Кинель",
    "Певек / Муниципальный округ Певек",
    "Город Оренбург",
    "Город Первомайск",
    "Зеленоградский / Зеленоградский г. о.",
    "Город Шарыпово",
    "Петрозаводский",
    "Светловский",
    "Невьянский",
    "Муниципальный район Заполярный район",
    "Реутов",
    "Суетский муниципальный район / Муниципальный округ Суетский район",
    "Серебряные Пруды",
    "Кашинский / Кашинский муниципальный район (с 2019 г. преобразован в Кашинский городской округ)",
    "Город Невинномысск",
    "Город Шарья",
    "Анжеро-Судженский",
    "Город Сасово",
    "Жигулевск",
    "Тисульский муниципальный округ / Тисульский муниципальный район",
    "Когалым",
    "Город Стрежевой",
    "Западнодвинский / Западнодвинский муниципальный район (с 2021 г. преобразован в Западнодвинский муниципальный округ)",
    "Орехово-Зуевский",
    "Урай",
    "Солнечногорск / Солнечногорский (до 2020 г.)",
    "Усть-Катавский",
]

In [ ]:
mapping = dict(zip(second, first))

def fast_replace(series, mapping):
    return series.map(mapping).fillna(series)

for col in ['mun_district', 'municipality', 'mo']:
    if col in df_wage.columns:
        df_wage[col] = fast_replace(df_wage[col], mapping)
    if col in df_popul.columns:
        df_popul[col] = fast_replace(df_popul[col], mapping)
    if col in df_sectors.columns:
        df_sectors[col] = fast_replace(df_sectors[col], mapping)

/tmp/ipykernel_3591/1609515327.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_popul[col] = fast_replace(df_popul[col], mapping)


#### Превращаем кумулятивную сумму по кварталам в значения по месяцам

In [ ]:
df_spend = df_spend.copy()
df_spend['period'] = pd.to_datetime(df_spend['period'])

df_spend_wide = (
    df_spend.pivot_table(
        index=['period', 'mo'],
        columns='category_15',
        values='value',
        aggfunc='sum'
    )
    .reset_index()
)
df_spend_wide.columns.name = None

df_wage = df_wage.copy()
df_wage['year'] = df_wage['year'].astype(int)

MONTHS = {
    'январь': 1, 'февраль': 2, 'март': 3,
    'апрель': 4, 'май': 5, 'июнь': 6,
    'июль': 7, 'август': 8, 'сентябрь': 9,
    'октябрь': 10, 'ноябрь': 11, 'декабрь': 12
}

def parse_cum_period(s):
    parts = s.lower().strip().split('-')
    start = MONTHS[parts[0].strip()]
    end = MONTHS[parts[-1].strip()]
    return start, end

df_wage[['start_month', 'end_month']] = df_wage['indicator_period'].apply(
    lambda x: pd.Series(parse_cum_period(x))
)

df_wage = df_wage.sort_values(['region_name', 'year', 'mun_district', 'end_month']).copy()

df_wage['prev_value'] = (
    df_wage.groupby(['region_name','year', 'mun_district'])['indicator_value']
       .shift(1)
)
df_wage['prev_end'] = (
    df_wage.groupby(['region_name','year', 'mun_district'])['end_month']
       .shift(1)
       .fillna(0)
)

df_wage['months_in_interval'] = df_wage['end_month'] - df_wage['prev_end']

df_wage['quarter_value'] = (
    df_wage['indicator_value'] * df_wage['end_month']
    - df_wage['prev_value'].fillna(0) * df_wage['prev_end']
) / df_wage['months_in_interval']

rows = []

for _, r in df_wage.iterrows():
    start_m = int(r['prev_end']) + 1
    end_m = int(r['end_month'])

    for m in range(start_m, end_m + 1):
        rows.append({
            'region_name': r['region_name'],
            'year': r['year'],
            'month': m,
            'period': pd.Timestamp(year=r['year'], month=m, day=1),
            'mun_district': r['mun_district'],
            'indicator_value': r['quarter_value']
        })

df_wage_monthly = pd.DataFrame(rows)

#### Джойн df_spend и df_wage

In [ ]:
df_spend_wage = df_spend_wide.merge(
    df_wage_monthly,
    left_on=['period', 'mo'],
    right_on=['period', 'mun_district'],
    how='inner'
)

df_spend_wage = df_spend_wage.drop(columns=['mun_district'])
df_spend_wage = df_spend_wage.rename(columns = {"indicator_value": "Средняя зарплата"})
df_spend_wage.head()

,period,mo,Все категории,Здоровье,Маркетплейсы,Общественное питание,Продовольствие,Транспорт,region_name,year,month,Средняя зарплата
0,2023-01-01,Абазинский муниципальный район,17866.0,1256.0,2058.0,1112.0,6075.0,1614.0,Карачаево-Черкесская Республика,2023,1,32052.8
1,2023-01-01,Абанский муниципальный район,17165.0,1250.0,1700.0,283.0,8545.0,1257.0,Красноярский край,2023,1,44206.3
2,2023-01-01,Абатский муниципальный район,19488.0,1076.0,1945.0,303.0,9042.0,1500.0,Тюменская область без автономных округов,2023,1,43148.1
3,2023-01-01,Абзелиловский муниципальный район,16642.0,954.0,638.0,746.0,7013.0,1474.0,Республика Башкортостан,2023,1,41343.8
4,2023-01-01,Абыйский муниципальный район,30207.0,877.0,840.0,194.0,13821.0,1430.0,Республика Саха (Якутия),2023,1,89695.4


In [ ]:
del df_spend_wage["period"]

In [ ]:
df_popul_small = df_popul.rename(columns={
    'indicator_value': 'population'
})[['region_name', 'mun_district', 'year', 'population', 'urban_share']].copy()

#### Предобрабатываем df_sectors, заменяем кумулятивную сумму по кварталам на значения по месяцам

In [ ]:
df_sectors_small = df_sectors[[
    'region_name', 'mun_district', 'year', 'indicator_period', 'okved2', 'indicator_value'
]].copy()

total_mask = df_sectors_small['okved2'] == 'Всего по обследуемым видам экономической деятельности'

df_sectors_total = df_sectors_small[total_mask].copy()

df_sectors_industries = df_sectors_small[~total_mask].copy()

MONTHS = {
    'январь': 1, 'февраль': 2, 'март': 3,
    'апрель': 4, 'май': 5, 'июнь': 6,
    'июль': 7, 'август': 8, 'сентябрь': 9,
    'октябрь': 10, 'ноябрь': 11, 'декабрь': 12
}

def parse_cum_period(s):
    parts = s.lower().strip().split('-')
    start = MONTHS[parts[0].strip()]
    end = MONTHS[parts[-1].strip()]
    return start, end

for df_ in (df_sectors_total, df_sectors_industries):
    df_[['start_month', 'end_month']] = df_['indicator_period'].apply(
        lambda x: pd.Series(parse_cum_period(x))
    )

def cum_to_monthly(df_in, value_col, group_cols):
    df_in = df_in.sort_values(group_cols + ['year', 'end_month']).copy()

    df_in['prev_value'] = (
        df_in.groupby(group_cols + ['year'])[value_col].shift(1)
    )
    df_in['prev_end'] = (
        df_in.groupby(group_cols + ['year'])['end_month'].shift(1)
    )

    first_mask = df_in['prev_end'].isna()
    first_jan_mask = first_mask & (df_in['start_month'] == 1)
    df_in.loc[first_jan_mask, 'prev_end'] = 0
    df_in.loc[first_jan_mask, 'prev_value'] = 0

    df_in['months_in_interval'] = df_in['end_month'] - df_in['prev_end']

    df_in['interval_sum'] = (
        df_in[value_col] * df_in['end_month']
        - df_in['prev_value'] * df_in['prev_end']
    )
    df_in['monthly_value'] = df_in['interval_sum'] / df_in['months_in_interval']

    df_in = df_in.dropna(subset=['prev_end', 'monthly_value']).copy()

    df_in['n_months'] = (df_in['end_month'] - df_in['prev_end']).astype(int)

    idx = np.repeat(df_in.index.values, df_in['n_months'].values)
    df_rep = df_in.loc[idx].copy()

    positions = np.arange(len(df_rep))
    starts = np.repeat(np.cumsum(np.r_[0, df_in['n_months'].values[:-1]]),
                       df_in['n_months'].values)
    df_rep['offset'] = positions - starts

    df_rep['month'] = df_rep['prev_end'].astype(int) + df_rep['offset'] + 1
    df_rep['period'] = pd.to_datetime(
        dict(year=df_rep['year'].astype(int),
             month=df_rep['month'],
             day=1)
    )
    df_rep['value'] = df_rep['monthly_value']

    keep = group_cols + ['year', 'month', 'period', 'value']
    return df_rep[keep].reset_index(drop=True)

df_sectors_industries_monthly = cum_to_monthly(
    df_sectors_industries,
    value_col='indicator_value',
    group_cols=['region_name','mun_district', 'okved2']
)

df_sectors_industries_wide = (
    df_sectors_industries_monthly
    .pivot_table(
        index=['region_name', 'mun_district', 'year', 'month', 'period'],
        columns='okved2',
        values='value',
        aggfunc='sum'
    )
    .reset_index()
)
df_sectors_industries_wide.columns.name = None

df_sectors_total_monthly = cum_to_monthly(
    df_sectors_total,
    value_col='indicator_value',
    group_cols=['mun_district', 'region_name']
)
df_sectors_total_monthly = df_sectors_total_monthly.rename(
    columns={'value': 'sector_total'}
)[['region_name', 'mun_district', 'year', 'month', 'period', 'sector_total']]

df_sectors_monthly = df_sectors_industries_wide.merge(
    df_sectors_total_monthly,
    on=['mun_district', 'region_name', 'year', 'month', 'period'],
    how='inner'
)

industry_cols = [c for c in df_sectors_monthly.columns
                 if c not in ['mun_district', 'region_name', 'year', 'month', 'period', 'sector_total']]

for c in industry_cols:
    df_sectors_monthly[f"share_{c}"] = df_sectors_monthly[c] / df_sectors_monthly['sector_total']

df_sectors_monthly = df_sectors_monthly.drop(columns=industry_cols + ['sector_total'])

#### Удаляем мусор, чтобы таблицы влезали в оперативную память

In [ ]:
del df_sectors_small, df_sectors_industries, df_sectors_industries_monthly
del df_sectors_total, df_sectors_total_monthly
gc.collect()

0

#### Финальный джойн

In [ ]:
df_spend_wage_popul = df_spend_wage.merge(
    df_popul_small,
    left_on=['region_name', 'mo', 'year'],
    right_on=['region_name', 'mun_district', 'year'],
    how='inner'
).drop(columns=['mun_district'])

df_spend_wage_popul = df_spend_wage_popul.copy()
df_spend_wage_popul['year'] = df_spend_wage_popul['year'].astype('Int64')
df_spend_wage_popul['month'] = df_spend_wage_popul['month'].astype('Int64')

df_sectors_monthly = df_sectors_monthly.copy()
df_sectors_monthly['year'] = df_sectors_monthly['year'].astype('Int64')
df_sectors_monthly['month'] = df_sectors_monthly['month'].astype('Int64')

df = df_spend_wage_popul.merge(
    df_sectors_monthly,
    left_on=['region_name', 'mo', 'year', 'month'],
    right_on=['region_name', 'mun_district', 'year', 'month'],
    how='left'
).drop(columns=['mun_district'])

In [ ]:
pd.set_option('display.max_columns', None)
df = df.drop(columns=['period'])
df.head()

,mo,Все категории,Здоровье,Маркетплейсы,Общественное питание,Продовольствие,Транспорт,region_name,year,month,Средняя зарплата,population,urban_share,share_Раздел C Обрабатывающие производства,"share_Раздел D Обеспечение электрической энергией, газом и паром, кондиционирование воздуха",share_Раздел F Строительство,"share_Раздел G Торговля оптовая и розничная, ремонт автотранспортных средств и мотоциклов",share_Раздел I Деятельность гостиниц и предприятий общественного питания,share_Раздел J Деятельность в области информации и связи,share_Раздел K Деятельность финансовая и страховая,share_Раздел L Деятельность по операциям с недвижимым имуществом,"share_Раздел M Деятельность профессиональная, научная и техническая",share_Раздел N Деятельность административная и сопутствующие дополнительные услуги,"share_Раздел O Государственное управление и обеспечение военной безопасности, социальное обеспечение",share_Раздел P Образование,share_Раздел Q Деятельность в области здравоохранения и социальных услуг,"share_Раздел R Деятельность в области культуры, спорта, организации досуга и развлечений",share_Раздел S Предоставление прочих видов услуг,share_Раздел U Деятельность экстерриториальных организаций и органов,"share_Раздел А Сельское, лесное хозяйство, охота, рыболовство и рыбоводство",share_Раздел В Добыча полезных ископаемых,"share_Раздел Е Водоснабжение, водоотведение, организация сбора и утилизация отходов, деятельность по ликвидации загрязнений",share_Раздел Н Транспортировка и хранение
0,Абыйский муниципальный район,30207.0,877.0,840.0,194.0,13821.0,1430.0,Республика Саха (Якутия),2023,1,89695.4,3786.0,0.503170,NaN,0.172881,NaN,0.016271,NaN,0.007458,NaN,NaN,NaN,NaN,0.123390,0.239322,0.100339,0.048136,NaN,NaN,NaN,NaN,NaN,0.247458
1,Агинский муниципальный район,20169.0,1146.0,1023.0,756.0,8097.0,2494.0,Забайкальский край,2023,1,39726.9,15742.0,0.309173,NaN,0.041026,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.280513,0.464103,NaN,0.029744,NaN,NaN,NaN,NaN,0.017436,NaN
2,Агрызский муниципальный район,18032.0,995.0,1044.0,630.0,7690.0,1167.0,Республика Татарстан,2023,1,50675.0,34692.0,0.572466,0.077392,0.046435,0.043636,0.028651,NaN,0.019266,0.004611,0.010538,NaN,0.005269,0.074757,0.176848,0.095999,0.030957,NaN,NaN,0.053680,0.016631,NaN,0.296394
3,Азнакаевский муниципальный район,20595.0,1087.0,1526.0,936.0,8748.0,1443.0,Республика Татарстан,2023,1,54833.4,57623.0,0.727158,0.033021,0.039054,0.015558,0.038101,0.004445,0.012939,0.005636,0.007303,0.010398,0.030243,0.062232,0.186617,0.088824,0.022543,NaN,NaN,0.066122,0.249643,0.002699,0.124544
4,Аксубаевский муниципальный район,14881.0,886.0,1291.0,244.0,7204.0,1195.0,Республика Татарстан,2023,1,39374.8,26484.0,0.342773,NaN,0.034755,NaN,0.019568,NaN,0.054907,0.004673,NaN,0.014311,NaN,0.099591,0.316881,0.139311,0.053738,NaN,NaN,0.087909,0.092582,NaN,0.036799


In [ ]:
industry_cols = [
    "share_Раздел В Добыча полезных ископаемых",
    "share_Раздел C Обрабатывающие производства",
    "share_Раздел D Обеспечение электрической энергией, газом и паром, кондиционирование воздуха",
    "share_Раздел Е Водоснабжение, водоотведение, организация сбора и утилизация отходов, деятельность по ликвидации загрязнений",
]

agriculture_cols = [
    "share_Раздел А Сельское, лесное хозяйство, охота, рыболовство и рыбоводство",
]

services_cols = [
    "share_Раздел F Строительство",
    "share_Раздел G Торговля оптовая и розничная, ремонт автотранспортных средств и мотоциклов",
    "share_Раздел Н Транспортировка и хранение",
    "share_Раздел I Деятельность гостиниц и предприятий общественного питания",
    "share_Раздел J Деятельность в области информации и связи",
    "share_Раздел K Деятельность финансовая и страховая",
    "share_Раздел L Деятельность по операциям с недвижимым имуществом",
    "share_Раздел M Деятельность профессиональная, научная и техническая",
    "share_Раздел N Деятельность административная и сопутствующие дополнительные услуги",
    "share_Раздел O Государственное управление и обеспечение военной безопасности, социальное обеспечение",
    "share_Раздел P Образование",
    "share_Раздел Q Деятельность в области здравоохранения и социальных услуг",
    "share_Раздел R Деятельность в области культуры, спорта, организации досуга и развлечений",
    "share_Раздел S Предоставление прочих видов услуг",
]

In [ ]:
df_result = df.copy()
df_result.fillna(0, inplace=True)
df_result["share_industry"] = df_result[industry_cols].sum(axis=1, min_count=1)
df_result["share_agriculture"] = df_result[agriculture_cols].sum(axis=1, min_count=1)
df_result["share_services"] = df_result[services_cols].sum(axis=1, min_count=1)

df = df_result.copy()
df.head()

,mo,Все категории,Здоровье,Маркетплейсы,Общественное питание,Продовольствие,Транспорт,region_name,year,month,Средняя зарплата,population,urban_share,share_Раздел C Обрабатывающие производства,"share_Раздел D Обеспечение электрической энергией, газом и паром, кондиционирование воздуха",share_Раздел F Строительство,"share_Раздел G Торговля оптовая и розничная, ремонт автотранспортных средств и мотоциклов",share_Раздел I Деятельность гостиниц и предприятий общественного питания,share_Раздел J Деятельность в области информации и связи,share_Раздел K Деятельность финансовая и страховая,share_Раздел L Деятельность по операциям с недвижимым имуществом,"share_Раздел M Деятельность профессиональная, научная и техническая",share_Раздел N Деятельность административная и сопутствующие дополнительные услуги,"share_Раздел O Государственное управление и обеспечение военной безопасности, социальное обеспечение",share_Раздел P Образование,share_Раздел Q Деятельность в области здравоохранения и социальных услуг,"share_Раздел R Деятельность в области культуры, спорта, организации досуга и развлечений",share_Раздел S Предоставление прочих видов услуг,share_Раздел U Деятельность экстерриториальных организаций и органов,"share_Раздел А Сельское, лесное хозяйство, охота, рыболовство и рыбоводство",share_Раздел В Добыча полезных ископаемых,"share_Раздел Е Водоснабжение, водоотведение, организация сбора и утилизация отходов, деятельность по ликвидации загрязнений",share_Раздел Н Транспортировка и хранение,share_industry,share_agriculture,share_services
0,Абыйский муниципальный район,30207.0,877.0,840.0,194.0,13821.0,1430.0,Республика Саха (Якутия),2023,1,89695.4,3786.0,0.503170,0.000000,0.172881,0.000000,0.016271,0.000000,0.007458,0.000000,0.000000,0.000000,0.000000,0.123390,0.239322,0.100339,0.048136,0.0,0.0,0.000000,0.000000,0.000000,0.247458,0.172881,0.000000,0.782373
1,Агинский муниципальный район,20169.0,1146.0,1023.0,756.0,8097.0,2494.0,Забайкальский край,2023,1,39726.9,15742.0,0.309173,0.000000,0.041026,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.280513,0.464103,0.000000,0.029744,0.0,0.0,0.000000,0.000000,0.017436,0.000000,0.058462,0.000000,0.774359
2,Агрызский муниципальный район,18032.0,995.0,1044.0,630.0,7690.0,1167.0,Республика Татарстан,2023,1,50675.0,34692.0,0.572466,0.077392,0.046435,0.043636,0.028651,0.000000,0.019266,0.004611,0.010538,0.000000,0.005269,0.074757,0.176848,0.095999,0.030957,0.0,0.0,0.053680,0.016631,0.000000,0.296394,0.140458,0.053680,0.786926
3,Азнакаевский муниципальный район,20595.0,1087.0,1526.0,936.0,8748.0,1443.0,Республика Татарстан,2023,1,54833.4,57623.0,0.727158,0.033021,0.039054,0.015558,0.038101,0.004445,0.012939,0.005636,0.007303,0.010398,0.030243,0.062232,0.186617,0.088824,0.022543,0.0,0.0,0.066122,0.249643,0.002699,0.124544,0.324417,0.066122,0.609382
4,Аксубаевский муниципальный район,14881.0,886.0,1291.0,244.0,7204.0,1195.0,Республика Татарстан,2023,1,39374.8,26484.0,0.342773,0.000000,0.034755,0.000000,0.019568,0.000000,0.054907,0.004673,0.000000,0.014311,0.000000,0.099591,0.316881,0.139311,0.053738,0.0,0.0,0.087909,0.092582,0.000000,0.036799,0.127336,0.087909,0.739778


In [ ]:
cols = ['region_name', 'mo', 'Все категории', 'Здоровье', 'Маркетплейсы',
       'Общественное питание', 'Продовольствие', 'Транспорт', 'year', 'month',
       'Средняя зарплата', 'population', 'urban_share', "share_industry",	"share_agriculture",	"share_services"
      ]
df = df[cols]

In [ ]:
df.head()

,region_name,mo,Все категории,Здоровье,Маркетплейсы,Общественное питание,Продовольствие,Транспорт,year,month,Средняя зарплата,population,urban_share,share_industry,share_agriculture,share_services
0,Республика Саха (Якутия),Абыйский муниципальный район,30207.0,877.0,840.0,194.0,13821.0,1430.0,2023,1,89695.4,3786.0,0.503170,0.172881,0.000000,0.782373
1,Забайкальский край,Агинский муниципальный район,20169.0,1146.0,1023.0,756.0,8097.0,2494.0,2023,1,39726.9,15742.0,0.309173,0.058462,0.000000,0.774359
2,Республика Татарстан,Агрызский муниципальный район,18032.0,995.0,1044.0,630.0,7690.0,1167.0,2023,1,50675.0,34692.0,0.572466,0.140458,0.053680,0.786926
3,Республика Татарстан,Азнакаевский муниципальный район,20595.0,1087.0,1526.0,936.0,8748.0,1443.0,2023,1,54833.4,57623.0,0.727158,0.324417,0.066122,0.609382
4,Республика Татарстан,Аксубаевский муниципальный район,14881.0,886.0,1291.0,244.0,7204.0,1195.0,2023,1,39374.8,26484.0,0.342773,0.127336,0.087909,0.739778


In [ ]:
df[['region_name', 'mo']].drop_duplicates().shape[0]

1094

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

In [ ]:
categories = [
    'Здоровье',
    'Маркетплейсы',
    'Общественное питание',
    'Продовольствие',
    'Транспорт'
]

for col in categories:
    df[f'{col}_share'] = df[col] / df['Все категории']

df = df.drop(columns=categories)

In [ ]:
df=df.drop(columns="Все категории")

In [ ]:
clean_rows_mask = ~np.isinf(df[["urban_share"]]).any(axis=1)
df = df[clean_rows_mask]

In [ ]:
df.columns

Index(['region_name', 'mo', 'year', 'month', 'Средняя зарплата', 'population',
       'urban_share', 'share_industry', 'share_agriculture', 'share_services',
       'Здоровье_share', 'Маркетплейсы_share', 'Общественное питание_share',
       'Продовольствие_share', 'Транспорт_share'],
      dtype='object')

In [ ]:
df.head()

,region_name,mo,year,month,Средняя зарплата,population,urban_share,share_industry,share_agriculture,share_services,Здоровье_share,Маркетплейсы_share,Общественное питание_share,Продовольствие_share,Транспорт_share
0,Республика Саха (Якутия),Абыйский муниципальный район,2023,1,1.125306,-0.411852,-0.180928,-0.064023,-0.682226,0.786951,-2.055774,-2.486268,-1.310460,0.059562,-0.562275
1,Забайкальский край,Агинский муниципальный район,2023,1,-0.822596,-0.315146,-0.825079,-0.782052,-0.682226,0.732683,0.950067,-1.985593,0.641425,-0.955532,5.149814
2,Республика Татарстан,Агрызский муниципальный район,2023,1,-0.395810,-0.161871,0.049167,-0.267494,-0.092981,0.817781,0.772640,-1.828799,0.481472,-0.502939,0.738466
3,Республика Татарстан,Азнакаевский муниципальный район,2023,1,-0.233705,0.023605,0.562807,0.886919,0.043588,-0.384484,0.513033,-1.474846,1.141929,-0.533721,1.138702
4,Республика Татарстан,Аксубаевский муниципальный район,2023,1,-0.836322,-0.228261,-0.713512,-0.349835,0.282746,0.498512,1.244209,-1.198230,-0.683664,0.540352,1.905018


In [ ]:
df.to_csv("df_sberne.csv", index=False)